<a href="https://colab.research.google.com/github/kithhooni-commits/ds-practice/blob/main/%EC%8B%A4%EC%8A%B55/colab_deconvolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kithhooni-commits/ds-practice/blob/main/%EC%8B%A4%EC%8A%B55/colab_deconvolution.ipynb)

# 실습5 Day 2 — Deconvolution (dipole)

1일차와 **같은 clean 이미지**를 쓰지만 열화가 다르다.

```
1일차   g = f + n            노이즈. 입력 24.67 dB
2일차   g = h * f            dipole 컨볼루션, 노이즈 없음. 입력 7.89 dB
```

그리고 결론이 정반대다. **노이즈가 없으면 역연산이 정확해서 고전 기법이 이긴다.**

| | 1일차 denoising | 2일차 deconvolution |
|---|---|---|
| 고전 최고 | adaptive 27.20 | **Wiener 118.54** |
| 딥러닝 | **DRUNet 37.42** | U-Net 25.59 |
| 승자 | 딥러닝 +10 dB | **고전 +93 dB** |

배포 예시 로그의 K 스윕이 `1e-4` 에서 멈춰 있어 이 격차가 16.7 dB 로만 보인다.
더 낮추면 93 dB 다. Day 3 강의자료가 경고한 *QSM Challenge 2.0 에서 고전 기법이
딥러닝을 이긴 사례* 가 여기서 그대로 재현된다.

## 0. 런타임 확인

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'GPU 없음 (이 노트북은 대부분 CPU 로 충분하다)'
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
torch 2.11.0+cu128 | cuda True


## 1. Drive 마운트

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. 데이터 준비

**2일차용으로 갱신된 dataset 을 써야 한다.** 1일차 것에는 `test_deconv_only` 와
`test_deconv_multi` 가 없다. zip 그대로 올려 뒀어도 아래 셀이 알아서 푼다.

In [3]:
import os
import zipfile
from pathlib import Path

SEARCH_ROOT = Path("/content/drive/MyDrive")
WORK = Path("/content/data")
DATA_ROOT = None

WANT = ("dataset", "code_deconvolution", "logs_deconvolution")


def looks_like_dataset(p: Path) -> bool:
    return (p / "train").is_dir() and (p / "test_label").is_dir()


WORK.mkdir(parents=True, exist_ok=True)
zips = [z for d in ("*.zip", "*/*.zip") for z in SEARCH_ROOT.glob(d)]
for z in sorted(zips):
    tag = next((w for w in WANT if z.name.startswith(w)), None)
    if tag is None:
        continue
    # dataset 은 2일차용으로 갱신됐을 수 있으니 항상 다시 푼다
    if (WORK / tag).exists() and tag != "dataset":
        continue
    print("푸는 중:", z.name)
    with zipfile.ZipFile(z) as f:
        f.extractall(WORK)

if DATA_ROOT is None:
    seen = [WORK] + [p for d in ('*', '*/*') for p in WORK.glob(d) if p.is_dir()]
    seen += [SEARCH_ROOT] + [p for d in ('*', '*/*') for p in SEARCH_ROOT.glob(d) if p.is_dir()]
    cands = [p for p in seen if looks_like_dataset(p)]
    if not cands:
        print("WORK 안:", [x.name for x in WORK.iterdir()])
        raise SystemExit("dataset 을 못 찾았다. DATA_ROOT 를 직접 적을 것")
    DATA_ROOT = cands[0]

DATA_ROOT = Path(DATA_ROOT)
os.environ["DS_DATA"] = str(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)
for sub in ("train", "val", "test_label", "test_deconv_only", "test_deconv_multi"):
    q = DATA_ROOT / sub
    n = len(list(q.glob("**/*.npy"))) if q.exists() else 0
    print(f"{'OK  ' if n else '없음'} {sub:<20} {n:>5} npy")

푸는 중: dataset-20260901T043818Z-1-001.zip
DATA_ROOT = /content/data/dataset
OK   train                 7268 npy
OK   val                    100 npy
OK   test_label             100 npy
OK   test_deconv_only       100 npy
OK   test_deconv_multi      100 npy


## 3. 코드 받기

In [4]:
REPO = Path("/content/ds-practice")
if REPO.exists():
    !cd "{REPO}" && git pull --ff-only
else:
    !git clone --depth 1 https://github.com/kithhooni-commits/ds-practice.git "{REPO}"

SRC = REPO / "실습5" / "src" / "deconv"
RUNS = Path("/content/runs")
RUNS.mkdir(exist_ok=True)
!ls "{SRC}"

Cloning into '/content/ds-practice'...
remote: Enumerating objects: 359, done.
remote: Counting objects: 100% (359/359), done.
remote: Compressing objects: 100% (302/302), done.
remote: Total 359 (delta 57), reused 288 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (359/359), 45.17 MiB | 74.01 MiB/s, done.
Resolving deltas: 100% (57/57), done.
baselines.py  learn_filter.py  run_baseline.py	 train_deconv.py
challenge.py  make_ppt2.py     run_challenge.py  unrolled.py
dcnet.py      metrics.py       run_multi.py
dipole.py     noise.py	       spectral.py


## 4. forward 모델이 배포 구현과 같은지 확인

1일차에 지표 구현을 0.0000 dB 까지 맞추고 시작했던 것과 같은 이유다.
우리가 만든 측정치가 배포 `ForwardSimulator` 의 출력과 같아야 이후 숫자가 의미를 갖는다.

`test_deconv_only` 는 배포된 측정치이므로 직접 대조할 수 있다.

In [21]:
import json
import numpy as np
import sys

sys.path.insert(0, str(SRC))
from challenge import forward

meta = json.loads((DATA_ROOT / "test_deconv_only" / "forward_meta.json").read_text())
errs = []
for r in meta[:20]:
    given = np.load(DATA_ROOT / "test_deconv_only" / r["file"]).astype(np.float64)
    gt = np.load(DATA_ROOT / "test_label" / r["file"]).astype(np.float64)
    errs.append(np.abs(given - forward(gt, tuple(r["B0_dir"]))).max())
print(f"배포 측정치 vs 우리 forward — 최대 절대차 {max(errs):.3e}")
print("→", "일치 (float32 정밀도 한계)" if max(errs) < 1e-5 else "어긋남, 확인 필요")
print("→ 차이가 0 이라는 것은 배포 측정치에 노이즈가 없다는 뜻이기도 하다")

배포 측정치 vs 우리 forward — 최대 절대차 1.871e-07
→ 일치 (float32 정밀도 한계)
→ 차이가 0 이라는 것은 배포 측정치에 노이즈가 없다는 뜻이기도 하다


## 5. 고전 기법 — K 를 얼마나 낮출 수 있는가

Wiener 는 `W = (1/D)·|D|²/(|D|²+K)` 다. `K → 0` 이면 직접 역산이 된다.
노이즈가 없으므로 K 를 낮출수록 계속 좋아진다 — 배포 스윕이 멈춘 `1e-4` 너머를 본다.

In [22]:
!cd "{SRC}" && python run_challenge.py --data "{DATA_ROOT}"

test_label 100장

방법                        PSNR     SSIM
---------------------------------------
입력 (blur)                7.892   0.0318
TKD t=0.15              32.617   0.9462
TKD t=0.05              38.203   0.9770
Tikhonov λ=1e-6         64.710   0.9999
Wiener K=1e-4           42.251   0.9881
Wiener K=1e-8           71.663   1.0000
Wiener K=1e-12         118.543   1.0000
Wiener 적응형 K            57.741   0.9988

(참고) 배포 U-Net 30 epoch  25.586 / 0.8779

측정치 노이즈 σ 별 PSNR (test 30장)

       σ    K=1e-02    K=1e-04    K=1e-06    K=1e-08    K=1e-12       적응형 K
---------------------------------------------------------------------------
   0e+00      25.47      41.79      55.00      71.21     118.84       55.49
   1e-05      25.47      41.79      54.51      57.45      47.91       53.90
   1e-04      25.47      41.56      46.61      37.93      27.91       45.53
   1e-03      25.45      36.00      28.19      17.94       7.91       32.11
   1e-02      23.71      18.30       8.23      -2.06    

## 6. `test_deconv_multi` — 커널 방향이 이미지마다 다르다

```
0_deg 25장 · 45_deg 25장 · 90_deg 25장 · 135_deg 25장
폴더끼리 이미지가 겹치지 않는다
```

**COSMOS 가 아니다.** 같은 장면을 여러 방향으로 찍은 게 아니라, 이미지마다 커널
방향이 다른 것이다. 그래서 여러 방향을 합쳐 0 영역을 메우는 건 불가능하고,
대신 **이미지마다 올바른 커널을 골라야** 한다.

`forward_meta.json` 에 방향이 적혀 있지만 채점 세트에 없을 수도 있다. 그래서
`estimate_b0` 로 **측정치만 보고 방향을 알아내는** 경로도 같이 잰다 —
`G = D·F` 이므로 |D|≈0 인 자리에는 에너지가 없고, 그 빈 방향을 찾으면 된다.

In [23]:
!cd "{SRC}" && python run_multi.py --data "{DATA_ROOT}" --estimate

test_deconv_only — 100장, B0 고정 (0, 1)

         K      PSNR     SSIM
-----------------------------
     1e-03     35.38   0.9703
     1e-04     42.25   0.9881
     1e-05     48.93   0.9967
     1e-06     56.07   0.9992
     1e-08     71.66   1.0000
     1e-12    109.72   1.0000

방향 추정 오차: 평균 0.192°  최대 0.530°

test_deconv_multi — 100장, 4방향 (겹치는 이미지 없음)

[메타 방향]
         K      0_deg     45_deg     90_deg    135_deg        ALL     SSIM
--------------------------------------------------------------------------
     1e-03      34.95      33.08      34.69      34.27      34.25   0.9627
     1e-04      40.92      38.94      40.99      39.27      40.03   0.9811
     1e-05      47.20      42.57      47.69      42.74      45.05   0.9923
     1e-06      54.19      42.18      55.06      42.63      48.52   0.9935
     1e-08      70.86      25.67      69.39      26.04      47.99   0.8845
     1e-12     109.56      24.66     108.80      25.04      67.02   0.8697

[추정 방향]
         K      0_deg     4

### 여기서 읽어야 할 것

- **방향을 틀리면 입력보다 나빠진다.** 0° 커널을 전부에 적용하면 −11 dB.
- **방향은 추정할 수 있다.** 메타데이터 없이 평균 0.15° 오차.
- **그런데 K→0 은 방향 오차에 극도로 취약하다.** 0.01° 만 틀려도 107 → 54 dB.
  45°/135° 는 방향이 정확해도 K=1e-12 에서 25 dB 밖에 안 나온다 — 격자와 0 영역이
  맞물려 1/D 가 수치적으로 폭발한다.
- 그래서 **적당한 K 가 정답**이다. 최소가 아니라.

## 7. 신경망을 쓰면?

입력을 무엇으로 주느냐가 이 문제의 핵심 설계 결정이다.

| `--input` | 네트워크가 하는 일 |
|---|---|
| `measure` | 역연산 **전체**를 배워야 한다. 배포 노트북 방식 |
| `wiener` | 역연산은 FFT 로 끝내고, 남은 국소 아티팩트만 지운다 |
| `both` | 둘 다 채널로 주고 알아서 섞게 한다 |

dipole 디컨볼루션은 원리적으로 **전역 연산**이다 (주파수 영역에서 1/D 곱하기).
한 픽셀을 복원하려면 이미지 전체를 봐야 하는데, DnCNN 의 수용영역은 35px,
U-Net 도 수십 px 다. `measure` 를 그대로 넣으면 구조적으로 불가능하다.

> 로컬 실험: `measure` 입력은 6 epoch 에 18.6 dB 에서 기어갔다.
> 같은 조건에서 `wiener` 입력은 200 iteration 만에 loss 가 17배 낮았다.

노이즈가 없으면 Wiener 만으로 118 dB 라 신경망이 개선할 여지가 없다.
`--noise` 로 측정치에 노이즈를 얹으면 그때부터 학습이 의미를 갖는다 —
**3일차가 정확히 그 조건**이다.

In [24]:
!cd "{SRC}" && python train_deconv.py \
    --model dncnn --features 64 \
    --input wiener \
    --noise 1e-3 --noise-random \
    --epochs 15 --patch 128 --batch 16 --lr 2e-4 \
    --workers 8 \
    --data "{DATA_ROOT}" \
    --out "{RUNS}" \
    --tag wiener_noisy

run    : /content/runs/0901-0705_deconv-wiener_wiener_noisy
model  : dncnn f64 | 입력 wiener (1ch) | loss charbonnier
노이즈 : σ=0.001 (매번 [0,σ] 에서 랜덤)
학습   : 7268장 patch 128 batch 16 -> 454 iter/ep, 15 ep
amp    : torch.bfloat16 | clip 1.0

  ep 00 it    0/454 loss 0.09299
  ep 00 it  100/454 loss 0.02426
  ep 00 it  200/454 loss 0.01869
  ep 00 it  300/454 loss 0.01659
  ep 00 it  400/454 loss 0.01538
[ep 00] loss 0.01467  val PSNR 35.573  SSIM 0.9338  11s  <- best
  ep 01 it    0/454 loss 0.00802
  ep 01 it  100/454 loss 0.00821
  ep 01 it  200/454 loss 0.00785
  ep 01 it  300/454 loss 0.00760
  ep 01 it  400/454 loss 0.00742
[ep 01] loss 0.00738  val PSNR 37.541  SSIM 0.9678  10s  <- best
  ep 02 it    0/454 loss 0.00756
  ep 02 it  100/454 loss 0.00661
  ep 02 it  200/454 loss 0.00662
  ep 02 it  300/454 loss 0.00661
  ep 02 it  400/454 loss 0.00655
[ep 02] loss 0.00651  val PSNR 39.164  SSIM 0.9770  10s  <- best
  ep 03 it    0/454 loss 0.00520
  ep 03 it  100/454 loss 0.00604
  ep 03

### 비교군 — 측정치를 그대로 넣었을 때

In [25]:
!cd "{SRC}" && python train_deconv.py \
    --model dncnn --features 64 \
    --input measure \
    --noise 1e-3 --noise-random \
    --epochs 15 --patch 128 --batch 16 --lr 2e-4 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag measure_noisy

run    : /content/runs/0901-0707_deconv-measure_measure_noisy
model  : dncnn f64 | 입력 measure (1ch) | loss charbonnier
노이즈 : σ=0.001 (매번 [0,σ] 에서 랜덤)
학습   : 7268장 patch 128 batch 16 -> 454 iter/ep, 15 ep
amp    : torch.bfloat16 | clip 1.0

  ep 00 it    0/454 loss 0.29428
  ep 00 it  100/454 loss 0.17961
  ep 00 it  200/454 loss 0.17141
  ep 00 it  300/454 loss 0.16331
  ep 00 it  400/454 loss 0.15675
[ep 00] loss 0.15453  val PSNR 15.336  SSIM 0.5561  10s  <- best
  ep 01 it    0/454 loss 0.14442
  ep 01 it  100/454 loss 0.13804
  ep 01 it  200/454 loss 0.13241
  ep 01 it  300/454 loss 0.12858
  ep 01 it  400/454 loss 0.12644
[ep 01] loss 0.12617  val PSNR 16.539  SSIM 0.6371  10s  <- best
  ep 02 it    0/454 loss 0.11208
  ep 02 it  100/454 loss 0.11477
  ep 02 it  200/454 loss 0.11362
  ep 02 it  300/454 loss 0.11168
  ep 02 it  400/454 loss 0.11179
[ep 02] loss 0.11130  val PSNR 18.360  SSIM 0.7099  10s  <- best
  ep 03 it    0/454 loss 0.09680
  ep 03 it  100/454 loss 0.10577
  ep

## 8. 정리

| | PSNR | SSIM |
|---|---|---|
| 입력 (blur) | 7.89 | 0.0318 |
| 배포 U-Net 30 epoch | 25.59 | 0.8779 |
| TKD t=0.05 | 38.20 | 0.9770 |
| Wiener K=1e-4 (배포 스윕의 끝) | 42.25 | 0.9881 |
| **Wiener K=1e-12** | **118.54** | **1.0000** |

노이즈가 없는 디컨볼루션에서 신경망은 해석적 역함수를 이길 수 없다.
그리고 그 답은 **극도로 취약하다** — σ=1e-3 짜리 미미한 노이즈만 얹혀도
K=1e-12 는 7.9 dB 로 무너져 흐린 입력과 같아진다.

3일차는 `g = h*f + n` 이다. 노이즈가 들어오는 순간 이 답은 못 쓰게 되고,
그때 비로소 학습이 필요해진다.

In [26]:
# ── ① 학습된 역필터 — 커널을 모른 채 데이터에서 찾는다 ──────────────
for n in (10, 50, 200, 1000):
    print(f"\n{'='*60}\n  train {n}장으로 필터 학습\n{'='*60}")
    !cd "{SRC}" && python learn_filter.py --data "{DATA_ROOT}" --n-train {n}



  train 10장으로 필터 학습
필터 학습 완료 — train 10장, 노이즈 σ=0.0
  |W| 범위 1.500 ~ 44285.3

test_deconv_only 100장
  PSNR 107.93   SSIM 1.0000
  해석적 1/D 와 상관 0.999985  ← 커널을 안 보고 커널의 역을 찾았다

저장 → /content/ds-practice/실습5/figures/learned_filter.json

  train 50장으로 필터 학습
필터 학습 완료 — train 50장, 노이즈 σ=0.0
  |W| 범위 1.500 ~ 44165.4

test_deconv_only 100장
  PSNR 108.54   SSIM 1.0000
  해석적 1/D 와 상관 0.999998  ← 커널을 안 보고 커널의 역을 찾았다

저장 → /content/ds-practice/실습5/figures/learned_filter.json

  train 200장으로 필터 학습
필터 학습 완료 — train 200장, 노이즈 σ=0.0
  |W| 범위 1.500 ~ 44097.3

test_deconv_only 100장
  PSNR 108.62   SSIM 1.0000
  해석적 1/D 와 상관 1.000000  ← 커널을 안 보고 커널의 역을 찾았다

저장 → /content/ds-practice/실습5/figures/learned_filter.json

  train 1000장으로 필터 학습
필터 학습 완료 — train 1000장, 노이즈 σ=0.0
  |W| 범위 1.500 ~ 44097.3

test_deconv_only 100장
  PSNR 108.65   SSIM 1.0000
  해석적 1/D 와 상관 1.000000  ← 커널을 안 보고 커널의 역을 찾았다

저장 → /content/ds-practice/실습5/figures/learned_filter.json


In [27]:
# ── ② 노이즈가 있으면 자동으로 Wiener 가 되는가 ────────────────────
for s in ("1e-4", "1e-3", "1e-2"):
    print(f"\n{'='*60}\n  σ = {s}\n{'='*60}")
    !cd "{SRC}" && python learn_filter.py --data "{DATA_ROOT}" --n-train 200 --noise {s}



  σ = 1e-4
필터 학습 완료 — train 200장, 노이즈 σ=0.0001
  |W| 범위 0.399 ~ 1149.7

test_deconv_only 100장
  PSNR 49.27   SSIM 0.9979
  해석적 1/D 와 상관 0.113698  ← 커널을 안 보고 커널의 역을 찾았다

저장 → /content/ds-practice/실습5/figures/learned_filter.json

  σ = 1e-3
필터 학습 완료 — train 200장, 노이즈 σ=0.001
  |W| 범위 0.007 ~ 186.2

test_deconv_only 100장
  PSNR 38.71   SSIM 0.9798
  해석적 1/D 와 상관 0.038383  ← 커널을 안 보고 커널의 역을 찾았다

저장 → /content/ds-practice/실습5/figures/learned_filter.json

  σ = 1e-2
필터 학습 완료 — train 200장, 노이즈 σ=0.01
  |W| 범위 0.000 ~ 37.1

test_deconv_only 100장
  PSNR 28.73   SSIM 0.8719
  해석적 1/D 와 상관 0.013255  ← 커널을 안 보고 커널의 역을 찾았다

저장 → /content/ds-practice/실습5/figures/learned_filter.json


In [28]:
import json
print(json.dumps(json.load(open("/content/ds-practice/실습5/figures/learned_filter.json")), indent=2, ensure_ascii=False))


{
  "n_train": 200,
  "noise": 0.01,
  "psnr": 28.72771379470825,
  "ssim": 0.8718865311145783,
  "corr_with_inverse_kernel": 0.013254626654088497
}


In [29]:
# ① 배포 U-Net, 우리 학습 레시피로 제대로 (크롭 없음, 전체 이미지)
!cd "{SRC}" && python train_deconv.py \
    --model unet --features 16 --input measure --noise 0 \
    --epochs 120 --batch 16 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag unet_full

# ② 같은 조건에 U-Net 을 키워서 (chans 16 → 48)
!cd "{SRC}" && python train_deconv.py \
    --model unet --features 48 --input measure --noise 0 \
    --epochs 120 --batch 8 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag unet48_full

# ③ DRUNet (수용영역이 가장 넓다)
!cd "{SRC}" && python train_deconv.py \
    --model drunet --features 64 --input measure --noise 0 \
    --epochs 60 --batch 4 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag drunet_full


run    : /content/runs/0901-0710_deconv-measure_unet_full
model  : unet f16 | 입력 measure (1ch) | loss charbonnier
노이즈 : σ=0.0
학습   : 7268장 patch 128 batch 16 -> 454 iter/ep, 120 ep
amp    : torch.bfloat16 | clip 1.0

  ep 00 it    0/454 loss 0.60967
  ep 00 it  100/454 loss 0.15884
  ep 00 it  200/454 loss 0.13288
  ep 00 it  300/454 loss 0.12022
  ep 00 it  400/454 loss 0.11383
[ep 00] loss 0.11166  val PSNR 20.240  SSIM 0.7980  4s  <- best
  ep 01 it    0/454 loss 0.09221
  ep 01 it  100/454 loss 0.08852
  ep 01 it  200/454 loss 0.08734
  ep 01 it  300/454 loss 0.08740
  ep 01 it  400/454 loss 0.08694
[ep 01] loss 0.08695  val PSNR 20.996  SSIM 0.8249  3s  <- best
  ep 02 it    0/454 loss 0.07001
  ep 02 it  100/454 loss 0.08451
  ep 02 it  200/454 loss 0.08397
  ep 02 it  300/454 loss 0.08444
  ep 02 it  400/454 loss 0.08292
[ep 02] loss 0.08288  val PSNR 21.921  SSIM 0.8460  3s  <- best
  ep 03 it    0/454 loss 0.07093
  ep 03 it  100/454 loss 0.07897
  ep 03 it  200/454 loss 0.080

In [30]:
!cd "{SRC}" && python train_deconv.py \
    --model spectral_dncnn --features 64 --input measure --noise 0 \
    --loss model_loss --model-loss-weight 0.8 \
    --epochs 60 --batch 8 --lr 1e-3 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag spectral_dncnn


[주의] spectral 은 크롭 학습을 못 한다. --patch 128 를 무시한다
run    : /content/runs/0901-0756_deconv-measure_spectral_dncnn
model  : spectral_dncnn f64 | 입력 measure (1ch) | loss model_loss
노이즈 : σ=0.0
학습   : 7268장 patch None batch 8 -> 908 iter/ep, 60 ep
amp    : torch.bfloat16 | clip 1.0

  ep 00 it    0/908 loss 0.13511
  ep 00 it  100/908 loss 0.17057
  ep 00 it  200/908 loss 0.16929
  ep 00 it  300/908 loss 0.16779
  ep 00 it  400/908 loss 0.16766
  ep 00 it  500/908 loss 0.16788
  ep 00 it  600/908 loss 0.16798
  ep 00 it  700/908 loss 0.16871
  ep 00 it  800/908 loss 0.16860
  ep 00 it  900/908 loss 0.16837
[ep 00] loss 0.16837  val PSNR 6.131  SSIM 0.0022  57s  <- best
  ep 01 it    0/908 loss 0.15658
  ep 01 it  100/908 loss 0.16040
  ep 01 it  200/908 loss 0.16355
  ep 01 it  300/908 loss 0.16754
  ep 01 it  400/908 loss 0.16965
  ep 01 it  500/908 loss 0.16980
  ep 01 it  600/908 loss 0.16938
  ep 01 it  700/908 loss 0.16950
  ep 01 it  800/908 loss 0.16907
  ep 01 it  900/908 loss 0.1682

In [32]:
# ④ 끝나면 자동으로 이어서 실행됨 (Colab 은 셀을 순서대로 큐에 넣는다)
!cd "{REPO}" && git pull --ff-only && git log --oneline -1

# ⑤ DC-Net
!cd "{SRC}" && python train_deconv.py \
    --model dcnet --refine unet --features 32 --tau 0.05 \
    --epochs 40 --batch 16 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag dcnet

# ⑥ Spectral + U-Net (커널 미사용)
!cd "{SRC}" && python train_deconv.py \
    --model spectral_unet --features 16 \
    --lr 3e-4 --lr-spectral 0.5 --warmup 200 \
    --epochs 60 --batch 16 --loss l2 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag spectral_unet

# 결과를 Drive 에 저장 — 런타임 끊겨도 남는다
import shutil, json, glob
from pathlib import Path
OUT = Path("/content/drive/MyDrive/실습프로젝트/runs_day2")
OUT.mkdir(parents=True, exist_ok=True)
for r in Path("/content/runs").glob("*"):
    shutil.copytree(r, OUT / r.name, dirs_exist_ok=True)

print("\n" + "="*54)
for h in sorted(glob.glob("/content/runs/*/history.json")):
    d = json.load(open(h))
    best = max(d, key=lambda x: x["val_psnr"])
    print(f"{Path(h).parent.name:<44} {best['val_psnr']:7.2f} / {best['val_ssim']:.4f}")


remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 9 (delta 5), reused 9 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 481.24 KiB | 16.59 MiB/s, done.
From https://github.com/kithhooni-commits/ds-practice
   2e6a014..93569bf  main       -> origin/main
Updating c1166fb..93569bf
error: Your local changes to the following files would be overwritten by merge:
	실습5/figures/learned_filter.json
Please commit your changes or stash them before you merge.
Aborting
usage: train_deconv.py [-h]
                       [--model {dncnn,unet,drunet,spectral,spectral_dncnn}]
                       [--input {measure,wiener,both}]
                       [--loss {l2,charbonnier,model_loss}]
                       [--model-loss-weight MODEL_LOSS_WEIGHT] [--noise NOISE]
                       [--noise-random] [--epochs EPOCHS] [--batch BATCH]
                       [--patch PATCH] [--lr LR] [--fe

In [5]:
!cd "{REPO}" && git pull --ff-only && git log --oneline -1   # ff01de8 확인

# ⑤ DC-Net (1번 팀 방식, 54.35 목표) — 가장 확실
!cd "{SRC}" && python train_deconv.py \
    --model dcnet --refine unet --features 32 --tau 0.05 \
    --epochs 40 --batch 16 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag dcnet

# ⑥ Spectral + U-Net (2번 팀 방식, 50.29 목표) — 커널 미사용
!cd "{SRC}" && python train_deconv.py \
    --model spectral_unet --features 16 \
    --lr 3e-4 --lr-spectral 0.5 --warmup 200 \
    --epochs 60 --batch 16 --loss l2 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag spectral_unet


Already up to date.
ff01de8 (grafted, HEAD -> main, origin/main, origin/HEAD) 실습5 Day2: SpectralFilter 가 0 에서 시작해 아무것도 학습 못 하던 버그 수정
run    : /content/runs/0902-0001_deconv-measure_dcnet
model  : dcnet f32 | 입력 measure (1ch) | loss charbonnier
노이즈 : σ=0.0
학습   : 7268장 patch None batch 16 -> 454 iter/ep, 40 ep
amp    : torch.bfloat16 | clip 1.0

  ep 00 it    0/454 loss 0.04702
  ep 00 it  100/454 loss 0.01759
  ep 00 it  200/454 loss 0.01530
  ep 00 it  300/454 loss 0.01409
  ep 00 it  400/454 loss 0.01331
[ep 00] loss 0.01301  val PSNR 38.918  SSIM 0.9808  16s  <- best
  ep 01 it    0/454 loss 0.00753
  ep 01 it  100/454 loss 0.01024
  ep 01 it  200/454 loss 0.01014
  ep 01 it  300/454 loss 0.00998
  ep 01 it  400/454 loss 0.00988
[ep 01] loss 0.00982  val PSNR 39.731  SSIM 0.9827  14s  <- best
  ep 02 it    0/454 loss 0.00796
  ep 02 it  100/454 loss 0.00900
  ep 02 it  200/454 loss 0.00907
  ep 02 it  300/454 loss 0.00918
  ep 02 it  400/454 loss 0.00910
[ep 02] loss 0.00916  val PS